<a href="https://colab.research.google.com/github/CarlosFranciscoM-Urzua/EJERCICIOS_PLN_COLAB/blob/main/Pr%C3%A1ctica_guiada_%C2%BFC%C3%B3mo_funciona_GPT%3F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica guiada: ¿Cómo funciona GPT?

**Montoya Urzúa Carlos Francisco**


Procesamiento del Lenguaje Natural | Ingeniería en Sistemas Computacionales

Modalidad: Google Colab | Duración sugerida: 100–120 minutos | Nivel: introductorio


## 1. Propósito y resultados de aprendizaje
Comprender el funcionamiento básico de GPT mediante la exploración de tokens, la predicción de la siguiente palabra y la generación de texto con un modelo preentrenado. Al terminar, el estudiante podrá ejecutar un modelo en Colab, modificar instrucciones y analizar los resultados.

# 2. ¿Qué es GPT y cómo funciona?
GPT significa Generative Pre-trained Transformer. Es una familia de modelos de lenguaje que aprende patrones estadísticos del texto durante un preentrenamiento y genera una secuencia prediciendo el siguiente token a partir del contexto anterior. Un token puede representar una palabra, parte de una palabra o un símbolo.
Flujo simplificado: texto de entrada → tokenización → representación numérica (embeddings y posiciones) → bloques Transformer con atención causal → probabilidades de tokens → selección del siguiente token → repetición. La atención ayuda al modelo a relacionar elementos del contexto. La máscara causal evita mirar tokens futuros al predecir.
Importante: GPT no consulta necesariamente hechos ni garantiza respuestas verdaderas. Puede generar errores. En esta práctica se utiliza GPT-2, un modelo pequeño y abierto, para estudiar el mecanismo; sus resultados no representan las capacidades de los GPT actuales.


## 3. Preparación de Google Colab (10 min)
1. Abrir un cuaderno nuevo en Google Colab. 2. Seleccionar Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (opcional). 3. Ejecutar las celdas en orden. La primera descarga de archivos requiere Internet. No se necesita clave API.


In [4]:
!pip -q install transformers torch

## 4. Actividad 1: reconocer los tokens (15 min)
Ejecuta el código. Observa que tokenizar no equivale siempre a separar por espacios. from transformers

In [6]:
from transformers import AutoTokenizer

modelo = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(modelo)
texto = "La inteligencia artificial ayuda a programar."
ids = tokenizer.encode(texto)
print("IDs:", ids)
print("Tokens:", tokenizer.convert_ids_to_tokens(ids))
print("Cantidad:", len(ids))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

IDs: [14772, 33649, 9324, 33743, 11666, 38762, 15339, 257, 1430, 283, 13]
Tokens: ['La', 'Ġintel', 'igen', 'cia', 'Ġartificial', 'Ġay', 'uda', 'Ġa', 'Ġprogram', 'ar', '.']
Cantidad: 11


Experimenta: cambia el texto por «Los estudiantes desarrollan aplicaciones» y luego por «Python 3.14».
Compara cantidad y forma de los tokens.

Pregunta: **¿por qué algunos fragmentos comienzan con un símbolo que indica espacio?**

Porque el modelo no elimina los espacios, los deja como el primer caracter del siguiente token.

In [7]:
texto = "Los estudiantes desarrollan aplicaciones"
ids = tokenizer.encode(texto)
print("IDs:", ids)
print("Tokens:", tokenizer.convert_ids_to_tokens(ids))
print("Cantidad:", len(ids))

IDs: [28903, 1556, 463, 3014, 274, 748, 283, 2487, 272, 257, 489, 291, 49443, 274]
Tokens: ['Los', 'Ġest', 'ud', 'iant', 'es', 'Ġdes', 'ar', 'roll', 'an', 'Ġa', 'pl', 'ic', 'acion', 'es']
Cantidad: 14


In [9]:
texto = "Python 3.14"
ids = tokenizer.encode(texto)
print("IDs:", ids)
print("Tokens:", tokenizer.convert_ids_to_tokens(ids))
print("Cantidad:", len(ids))

IDs: [37906, 513, 13, 1415]
Tokens: ['Python', 'Ġ3', '.', '14']
Cantidad: 4




# 5. Actividad 2: predecir el siguiente token (20 min)


GPT asigna una puntuación a cada token posible. Convertimos las puntuaciones a probabilidades y mostramos cinco candidatos.


In [11]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

prompt = "The computer can"
entradas = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    salida = model(**entradas)

prob = torch.softmax(salida.logits[0, -1], dim=-1)
valores, indices = torch.topk(prob, 5)

for indice, valor in zip(indices, valores):
    print(repr(tokenizer.decode([indice.item()])),           round(valor.item(), 4))


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

' be' 0.0975
' also' 0.068
' read' 0.0421
' then' 0.0249
"'t" 0.0235


Observación: GPT-2 fue entrenado principalmente con inglés; por ello usamos una frase inglesa en esta actividad. Experimenta: reemplaza el prompt por «The student learns» y compara el orden de las probabilidades. La lista de candidatos depende del contexto.

In [13]:
prompt = "The student learns"
entradas = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    salida = model(**entradas)

prob = torch.softmax(salida.logits[0, -1], dim=-1)
valores, indices = torch.topk(prob, 5)

for indice, valor in zip(indices, valores):
    print(
        repr(tokenizer.decode([indice.item()])),
        round(valor.item(), 4)
    )


' to' 0.1978
' that' 0.1772
' the' 0.1054
' how' 0.0947
' about' 0.0935




## 6. Actividad 3: generar texto (20 min)
Ahora generaremos texto token por token. La temperatura modifica la distribución de probabilidad: valores menores suelen producir resultados más conservadores; mayores, más variados.


In [14]:
from transformers import set_seed

set_seed(42)

prompt = "Artificial intelligence is useful because"

entradas = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    generado = model.generate(
        **entradas,
        max_new_tokens=45,
        do_sample=True,
        temperature=0.8,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(generado[0], skip_special_tokens=True))


Artificial intelligence is useful because it allows us to create new algorithms and more efficient models, to see what's happening in the future. The AI's predictions are always correct.

The problem with artificial intelligence is that it's limited. We can only


**Experimenta: ejecuta primero con temperatura 0.3 y después con 1.2 (con la misma semilla antes de cada generación). Describe qué cambia. Si aparecen frases extrañas, recuerda que el modelo completa texto: no está siguiendo instrucciones como un asistente moderno.**

Las frases cambian, aunque no aparecen frases extrañas.

In [15]:
from transformers import set_seed

set_seed(42)

prompt = "Artificial intelligence is useful because"

entradas = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    generado = model.generate(
        **entradas,
        max_new_tokens=45,
        do_sample=True,
        temperature=0.3,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(generado[0], skip_special_tokens=True))


Artificial intelligence is useful because it allows us to create new things, and to create new things that are not already there.

The problem with artificial intelligence is that it is not really a problem at all. It is a problem of how to create


In [16]:
from transformers import set_seed

set_seed(42)

prompt = "Artificial intelligence is useful because"

entradas = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    generado = model.generate(
        **entradas,
        max_new_tokens=45,
        do_sample=True,
        temperature=1.2,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(generado[0], skip_special_tokens=True))


Artificial intelligence is useful because it offers more options, and in turn more resources: Google is moving toward a new type of artificial intelligence, AI.

But what sort of AI is being experimented with will soon become more complicated. We could find a



## 7. Actividad 4: comparar generación determinista y muestreo (15 min)


In [17]:
texto = "In the future, computers will"
e = tokenizer(texto, return_tensors="pt")

with torch.no_grad():
    greedy = model.generate(
        **e,
        max_new_tokens=35,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id)

set_seed(7)

with torch.no_grad():
    muestra = model.generate(
        **e,
        max_new_tokens=35,
        do_sample=True,
        top_p=0.9,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

print("Greedy:", tokenizer.decode(greedy[0]))

print("Muestreo:", tokenizer.decode(muestra[0]))


Greedy: In the future, computers will be able to do things like send and receive messages, and they will be able to do things like send and receive messages, and they will be able to do things like send
Muestreo: In the future, computers will only be able to read text and words. These words and phrases will need to be interpreted to understand what we are saying and doing, and to understand what we're thinking and


**Analiza: ¿cuál de las dos salidas parece más repetitiva? ¿Cuál ofrece mayor variedad? Explica que greedy escoge el token más probable en cada paso, mientras que el muestreo selecciona entre alternativas.**


La salida más repetitiva es la de la generación **Greedy** (determinista). En el ejemplo, podemos observar cómo repite de forma buclada la frase *"and they will be able to do things like send and receive messages"*.

La salida con mayor variedad es la del **Muestreo** (sampling), la cual fluye de manera mucho más natural y propone ideas diversas sobre interpretar palabras, frases y pensamientos.

**Explicación del mecanismo:**
- **Búsqueda Greedy (Codiciosa):** Selecciona de manera estricta el token con la probabilidad más alta en cada paso. Aunque es eficiente y lógico a corto plazo, suele provocar que el modelo caiga en bucles repetitivos y predecibles al engancharse en secuencias de alta probabilidad común.
- **Muestreo (Sampling):** En lugar de elegir siempre el primer candidato, selecciona al azar un token de entre un conjunto de alternativas altamente probables (filtradas por parámetros como `top_p` o `temperature`). Esto introduce variabilidad y creatividad, evitando la monotonía a cambio de asumir un pequeño riesgo de perder coherencia si la temperatura es demasiado alta.


## 8. Actividad 5: mini reto aplicado a PLN (15 min)
Crea tres prompts en inglés sobre temas distintos: programación, educación y seguridad informática. Genera 30 tokens para cada uno con temperatura 0.7 y top_k=40. Registra el texto inicial, la salida, una observación sobre coherencia y una posible inexactitud. Evita introducir datos personales.


In [21]:
from transformers import set_seed
import torch
from transformers import AutoModelForCausalLM

set_seed(42)

prompts = [
    ("Programacion", "Learning programming in the AI era"),
    ("Educación", "Education in Mexico compared to other countries"),
    ("Seguridad", "Advices for informatic security"),
]

for tema, prompt in prompts:
    entradas = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        generado = model.generate(
            **entradas,
            max_new_tokens=45,
            do_sample=True,
            temperature=0.7,
            top_k=40,
            pad_token_id=tokenizer.eos_token_id
        )

    print("\nTEMA: ", tema, "\nPROMPT: ", prompt)
    print(tokenizer.decode(generado[0], skip_special_tokens=True))



TEMA:  Programacion 
PROMPT:  Learning programming in the AI era
Learning programming in the AI era: What does a programming language have to do with machine learning?

Jordie Johnson: For machine learning, I think it's pretty much an imperative, you know, that is to provide the most robust and complete

TEMA:  Educación 
PROMPT:  Education in Mexico compared to other countries
Education in Mexico compared to other countries, has been relatively low.

"It is quite a shock when you look at the number of people who are out there seeking to become citizens," said Guglielmi. "There is no shortage."


TEMA:  Seguridad 
PROMPT:  Advices for informatic security
Advices for informatic security policy

In the past two years there have been several changes to the way that organisations interact with one another in order to avoid problems or to ensure that they are not compromised.

In particular the new Privacy Policy (


En este caso sí se mostraron muchas frases incoherentes o fuera de contexto.

## 9. Preguntas de reflexión y evidencias
Responde:

**(1) ¿Qué diferencia hay entre palabra y token?**
- Una **palabra** es una unidad lingüística con significado propio dentro de un idioma. Un **token** es la unidad mínima de procesamiento para un modelo de lenguaje. Los tokens pueden ser palabras completas, partes de palabras (subpalabras como sílabas o raíces) o caracteres individuales (incluidos espacios y signos de puntuación). Por ejemplo, la palabra "desarrollan" puede dividirse en varios tokens como `['Ġdes', 'ar', 'roll', 'an']`.

**(2) ¿Para qué sirve la atención causal?**
- Sirve para asegurar que el modelo solo pueda "mirar" o prestar atención a los tokens que ya han aparecido antes (el contexto izquierdo) al predecir el siguiente token. Esto se logra mediante una máscara que bloquea la información de los tokens futuros, garantizando que el proceso de generación sea estrictamente autorregresivo y no haga "trampa" mirando hacia adelante.

**(3) ¿Por qué una misma entrada puede generar distintas salidas?**
- Porque cuando se utiliza el método de **muestreo (sampling)**, el modelo no elige de manera fija el token con la probabilidad más alta. En su lugar, utiliza una distribución de probabilidad para seleccionar aleatoriamente el siguiente token de entre un conjunto de opciones viables. Esta naturaleza estocástica (azar controlado) hace que cada ejecución genere un camino o texto diferente a partir de la misma entrada.

**(4) ¿Qué efecto tiene la temperatura?**
- La temperatura controla el nivel de aleatoriedad y creatividad en la generación:
  - **Valores bajos (cercanos a 0):** Concentran la probabilidad en los tokens más seguros y lógicos, haciendo la salida determinista, coherente pero potencialmente repetitiva o plana.
  - **Valores altos (mayores a 1.0):** Aplanan la distribución de probabilidad, dando más oportunidades a tokens menos comunes. Esto genera un texto más creativo y variado, pero también incrementa el riesgo de incoherencias, errores gramaticales o frases sin sentido.

**(5) ¿Por qué hay que verificar la información generada?**
- Porque los modelos GPT funcionan mediante predicción estadística del lenguaje, no consultando una base de datos de hechos reales. El modelo genera secuencias de palabras que *suenan bien y parecen coherentes*, pero esto no garantiza que sean verdaderas. Este fenómeno se conoce como **alucinación**, donde el modelo puede inventar datos, citas o información técnica errónea de manera muy convincente.


Entregable: enlace al cuaderno de Colab o archivo .ipynb con las cinco actividades ejecutadas, capturas o salidas, respuestas a las preguntas y una conclusión de 5 a 8 líneas.

## 10. Criterios de evaluación (100 puntos)
Ejecución correcta de celdas: 30 puntos. Comparación y experimentación: 25 puntos. Explicación del mecanismo GPT: 20 puntos. Respuestas de reflexión: 15 puntos. Orden, comentarios y conclusión: 10 puntos.

## 11. Problemas frecuentes
ModuleNotFoundError: vuelve a ejecutar la instalación y reinicia el entorno si es necesario. Descarga lenta: espera a que termine la primera carga del modelo. Memoria insuficiente: reinicia el entorno, reduce max_new_tokens y evita cargar varios modelos. Advertencia de pad_token_id: ya se especifica el token de fin de secuencia en los ejemplos.
# 12. Para profundizar
Conceptos clave: tokenización subléxica, embeddings, autoatención, Transformer decodificador, distribución softmax, temperatura, top-k y top-p. La práctica estudia inferencia con un modelo preentrenado; no entrena un GPT desde cero ni realiza ajuste fino.
## Referencias:
Vaswani et al. (2017), «Attention Is All You Need»; Radford et al. (2019), «Language Models are Unsupervised Multitask Learners»; documentación de Hugging Face Transformers, apartados de generación y modelos causales.
